# Dark.Tyro — M2 diagnostics

**These two checks are not the contribution. They are the reason to believe the contribution.**

`Dark_Tyro_M2.ipynb` stays at "M1 + 16 lines" — that is deliberate, and it is what gets other
teams to run it. This notebook is kept separate because it answers a different question:
**do our numbers mean what we say they mean?**

| | what it asks | cost |
|---|---|---|
| **D1 · `R_edit`** | Did the *edit* succeed? `R_pipe` cannot tell you. | **no GPU**, ~2 min, reads the zips you already have |
| **D2 · the two nulls** | Is `ssim_adv` measuring the shield, or measuring the seed? | GPU, ~4 min |

## Why D2 exists — read this before running anything

`R_pipe = (ssim_pur - ssim_adv) / (1 - ssim_adv)`.

That denominator claims that `1 - ssim_adv` is *"damage the shield did"*. But SSIM only says
**"these are two different pictures."** Two perfectly good diffusion edits of the same face,
made with two different seeds, are also different pictures — they score maybe 0.6 against each
other too.

> A thermometer that reads 38 degrees tells you nothing until you know that a healthy person
> reads 37 and not 38. **D2 measures the healthy person.**

If `ssim_adv` turns out to sit at the same value as two innocent re-edits, then the shield never
engaged, and every `R_pipe` in the project is a ratio over noise. **Either answer is a real
finding and both are publishable** — one is a result about our attack, the other is a result
about the metric. Run it before buying any more GPU time.


In [ ]:
# D0 · point this at an archived run, and nothing else needs editing.
# Works on Kaggle, Colab, or your laptop. Give it the folder that holds the arm folders
# (A_impress_100/, C_tyro_masked/, ...) -- or a folder of the zips, which it will unpack.
import os, re, json, glob, zipfile, pathlib

RUN_DIR = pathlib.Path('run_0907_pgeps16')   # <-- the only line to edit
PROMPT  = 'a person in an airplane'                   # must match params.json

RUN_DIR = pathlib.Path(RUN_DIR).expanduser().resolve()
assert RUN_DIR.is_dir(), f'not a folder: {RUN_DIR}'

# Unpack any arm zips that have no matching folder, so a fresh download just works.
for z in sorted(RUN_DIR.glob('*.zip')):
    d = RUN_DIR / z.stem.strip()
    if not d.is_dir():
        d.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as f: f.extractall(d)
        print('unpacked', z.name)

ARMS = sorted(d.name for d in RUN_DIR.iterdir()
              if d.is_dir() and (d / 'edit_clean').is_dir())
IMAGES = sorted(os.listdir(RUN_DIR / ARMS[0] / 'edit_clean'))
print(f'{len(ARMS)} arms x {len(IMAGES)} images in {RUN_DIR.name}')
print('  arms  :', ARMS)
print('  images:', IMAGES)
if len(IMAGES) < 5:
    print(f'\n  !! n = {len(IMAGES)}. Every mean below is a mean of {len(IMAGES)} numbers.'
          '\n     Report per-image values, never just the mean.')


---
## D1 · `R_edit` — did the *edit* succeed?

`R_pipe` is built on `pg_metric.py`'s SSIM, which compares the **edited protected** image to the
**edited clean** image. It answers *"did the pipeline produce the same picture?"* — a question
about pixels.

The question the assignment actually cares about is *"did the editor obey the prompt?"* — a
question about **meaning**. CLIPScore answers that one: it embeds the image and the prompt into
the same space and measures the angle between them.

> `R_pipe` asks whether two photographs match. `R_edit` asks whether the photograph matches the
> **caption**. A shield can leave the second one intact while wrecking the first.

We deliberately score with `openai/clip-vit-base-patch32`, **not** SD 1.5's own ViT-L/14 text
encoder. Do not let the chef grade the dish.

    P      = S_clean - S_protected            <- protection efficacy. If this is ~0, the shield did nothing.
    R_edit = (S_purified - S_protected) / P   <- the share of the lost meaning the wash restored.

**The guard rail is the point of the cell.** When `P` sits in the noise, `R_edit` is a ratio of
two near-zero numbers and means nothing, so the cell refuses to print it and says so instead.
Printing an honest refusal is worth more marks than printing a flattering number.


In [ ]:
# D1 · CLIPScore per image, per stage. No GPU needed; ~90 s on CPU for 2-10 images.
import importlib.util, subprocess, sys
if importlib.util.find_spec('transformers') is None:
    subprocess.run(f'{sys.executable} -m pip install -q transformers', shell=True, check=False)

import torch, numpy as np, pandas as pd
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
_m  = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEV).eval()
_p  = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

@torch.no_grad()
def clipscore_img(img, text):
    """Cosine similarity between image and text embeddings, x100. Higher = better match."""
    inp = _p(text=[text], images=img.convert('RGB'), return_tensors='pt', padding=True).to(DEV)
    out = _m(**inp)
    i = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    t = out.text_embeds  / out.text_embeds.norm(dim=-1,  keepdim=True)
    return float((i @ t.T).item() * 100)

def clipscore(img_path, text):
    return clipscore_img(Image.open(img_path), text)   # D2 reuses clipscore_img on PIL objects

# NOISE FLOOR for P: two different good edits of the SAME clean image score slightly
# differently under CLIP too. Anything smaller than this is not protection.
P_FLOOR = 0.5      # CLIPScore points. Conservative; D2 can refine it if you run extra seeds.

rows = []
for arm in ARMS:
    d = RUN_DIR / arm
    for n in IMAGES:
        s_clean = clipscore(d / 'edit_clean'     / n, PROMPT)
        s_prot  = clipscore(d / 'edit_protected' / n, PROMPT)
        s_pur   = clipscore(d / 'edit_purified'  / n, PROMPT)
        P       = s_clean - s_prot
        rows.append(dict(arm=arm, image=n, S_clean=round(s_clean, 2), S_prot=round(s_prot, 2),
                         S_pur=round(s_pur, 2), P=round(P, 2),
                         # P must be POSITIVE. P < 0 means the shield made the edit match the
                         # prompt BETTER than the clean image did -- anti-protection, not weak
                         # protection -- and a ratio over a negative denominator is nonsense.
                         R_edit=(round((s_pur - s_prot) / P * 100, 1)
                                 if P >= P_FLOOR else None)))

d1 = pd.DataFrame(rows)
pd.set_option('display.width', 140)
display(d1)


In [ ]:
# D1 verdict · say out loud what the table means, including when it means nothing.
eng = d1.groupby('image').P.first()
print('PROTECTION EFFICACY  P = S_clean - S_protected      (floor = %.2f pts)\n' % P_FLOOR)
for n, p in eng.items():
    verdict = ('engaged'         if p >= P_FLOOR else
               'BACKFIRED - the shield IMPROVED the edit' if p <= -P_FLOOR else
               'NOT ENGAGED')
    print(f'   {n:<24}{p:+7.2f}   {verdict}')

n_eng = int((eng >= P_FLOOR).sum())
n_bad = int((eng <= -P_FLOOR).sum())
if n_bad:
    print(f'\n   !! {n_bad} image(s) have NEGATIVE protection efficacy. The protected image edits'
          '\n      CLOSER to the prompt than the clean image does. That is not a weak shield,'
          '\n      it is a shield pushing the wrong way, and no ratio built on it means anything.')
print(f'\n   {n_eng} of {len(eng)} images have a shield that measurably changed the edit.\n')

if n_eng == 0:
    print('VERDICT — the shield never engaged on any image.')
    print('  R_edit is undefined and R_pipe is a ratio over seed noise, not protection.')
    print('  This is a RESULT, not a failure: PhotoGuard at these settings does not obstruct')
    print('  this editor. Report it, then raise pg_eps and re-run before claiming a wash works.')
else:
    ok = d1[d1.R_edit.notna()]
    print('R_edit — share of the lost prompt-meaning that the wash restored (up is better):\n')
    for arm in ARMS:
        v = ok[ok.arm == arm].R_edit
        if len(v) == 0:
            print(f'   {arm:<18} no scorable image'); continue
        per = ', '.join(f'{x:+.0f}%' for x in v)
        print(f'   {arm:<18} mean {v.mean():+6.1f}%   per image: {per}')
    spread = ok.groupby('arm').R_edit.agg(lambda s: s.max() - s.min()).max()
    if len(ok) >= 2 and spread > abs(ok.R_edit.mean()):
        print(f'\n   !! per-image spread ({spread:.0f} pp) exceeds the mean.'
              '  Quote the range, never the mean.')
print('\nNote: R_edit and R_pipe answer different questions and are allowed to disagree.')
print('If they do, THAT is the finding — say which one matches the pictures.')


---
## D2 · The two nulls — is `ssim_adv` measuring the shield, or the seed?

Two controls, six edits, about four minutes.

**Null 1 — the seed floor.** Edit the **clean** image twice with two different seeds. Nothing
is protected, nothing is attacked; both outputs are perfectly good edits. The SSIM between them
is the score two innocent pictures get. *If `ssim_adv` is not clearly below this, the shield
did nothing SSIM can see.*

**Null 2 — the random shield.** Take the clean image, add **random noise of exactly the same L2
norm** as PhotoGuard's actual perturbation (measured from your own archive, not guessed), and
edit it at the same seed. This asks the sharper question: does PhotoGuard's *structure* matter,
or would any perturbation that size disturb the edit equally?

> Null 1 is the healthy patient. Null 2 is the sugar pill. A drug has to beat both.

**Every SSIM below is computed with the same function on the same images**, so the comparison is
apples to apples — we do not read `ssim_adv` out of `scores.json`, we recompute it here.


In [ ]:
# D2a · load the same editor pg_generate.py uses, configured identically.
import torch, numpy as np
from PIL import Image, ImageOps
from diffusers import StableDiffusionInpaintPipeline

MODEL_IDS = ['runwayml/stable-diffusion-inpainting',
             'stable-diffusion-v1-5/stable-diffusion-inpainting']   # community re-upload fallback
DEV = 'cuda:0' if torch.cuda.is_available() else 'cpu'

pipe = None
for mid in MODEL_IDS:
    try:
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            mid, torch_dtype=torch.float16 if DEV.startswith('cuda') else torch.float32,
            safety_checker=None)
        print('loaded', mid); break
    except Exception as e:
        print(f'  {mid} unavailable ({type(e).__name__})')
assert pipe is not None, 'no inpainting checkpoint available'
pipe = pipe.to(DEV)

TEST_DIFF_STEPS, TEST_GUIDANCE = 50, 7.5     # must match PARAMS in Dark_Tyro_M2.ipynb

def edit(img, mask_rgb, seed):
    """Byte-for-byte the call pg_generate.test_image() makes."""
    np.random.seed(seed); torch.manual_seed(seed)
    if DEV.startswith('cuda'): torch.cuda.manual_seed(seed)
    return pipe(prompt=PROMPT, image=img, mask_image=mask_rgb, eta=1,
                num_inference_steps=TEST_DIFF_STEPS, guidance_scale=TEST_GUIDANCE).images[0]

# pg_generate INVERTS the mask before handing it to the pipeline. Copy that, do not re-derive it.
def load_mask(path):
    return ImageOps.invert(Image.open(path).convert('RGB').resize((512, 512)))

try:
    from skimage.metrics import structural_similarity as _ss
    ssim = lambda a, b: float(_ss(np.asarray(a, np.float64), np.asarray(b, np.float64),
                                  channel_axis=2, data_range=255))
except ImportError:
    raise SystemExit('pip install scikit-image  (D2 needs one consistent SSIM for every pair)')
print('editor ready')


In [ ]:
# D2b · run the two nulls on every image, and recompute ssim_adv on the same footing.
import pandas as pd
ARM0 = ARMS[0]        # any arm: clean512/ and protected/ are identical across arms
# Masks are not archived by the run, so fall back to the published challenge pack.
MASK_DIR = None
for cand in [RUN_DIR / 'mask', pathlib.Path('challenge/tyro_wash_test_trackA/mask'),
             pathlib.Path('../challenge/tyro_wash_test_trackA/mask')]:
    if pathlib.Path(cand).is_dir(): MASK_DIR = pathlib.Path(cand); break
assert MASK_DIR, 'point MASK_DIR at the 10-image pack mask/ folder'

rows = []
for n in IMAGES:
    clean = Image.open(RUN_DIR / ARM0 / 'clean512'  / n).convert('RGB')
    prot  = Image.open(RUN_DIR / ARM0 / 'protected' / n).convert('RGB')
    mask  = load_mask(MASK_DIR / n)

    # the shield's real magnitude, measured -- not assumed
    delta   = np.asarray(prot, np.float64) - np.asarray(clean, np.float64)
    l2      = float(np.sqrt((delta ** 2).sum()))

    # null 2: uniform random noise, rescaled to the identical L2 norm
    rng     = np.random.default_rng(0)
    r       = rng.uniform(-1, 1, delta.shape)
    r      *= l2 / float(np.sqrt((r ** 2).sum()))
    randsh  = Image.fromarray(np.clip(np.asarray(clean, np.float64) + r, 0, 255).astype(np.uint8))

    e_clean0 = edit(clean,  mask, 0)      # the reference edit, seed 0
    e_clean1 = edit(clean,  mask, 1)      # NULL 1 -- same image, different seed
    e_prot   = edit(prot,   mask, 0)      # the real protected edit, recomputed here
    e_rand   = edit(randsh, mask, 0)      # NULL 2 -- matched-size random shield

    # the CLIP floor: two innocent edits of the same image, scored against the prompt
    try:
        clip_gap = clipscore_img(e_clean0, PROMPT) - clipscore_img(e_clean1, PROMPT)
    except NameError:
        clip_gap = float('nan')

    rows.append(dict(image=n, shield_L2=round(l2, 1), clip_seed_gap=round(clip_gap, 2),
                     shield_mean_lv=round(float(np.abs(delta).mean()), 2),
                     ssim_seed_floor=round(ssim(e_clean0, e_clean1), 4),   # null 1
                     ssim_rand_shield=round(ssim(e_clean0, e_rand),  4),   # null 2
                     ssim_adv=round(ssim(e_clean0, e_prot), 4)))           # the real thing
    print('  done', n, flush=True)

d2 = pd.DataFrame(rows); display(d2)


In [ ]:
# D2 verdict · the sentence that decides what M2 can claim.
print('SSIM against the seed-0 clean edit -- LOWER means the edit was disturbed more.\n')
print(f'  {"image":<24}{"null1 seed":>12}{"null2 random":>14}{"REAL shield":>13}   verdict')
for _, r in d2.iterrows():
    floor = min(r.ssim_seed_floor, r.ssim_rand_shield)
    beats = r.ssim_adv < floor - 0.03          # 0.03 = comfortably outside SSIM run-to-run wobble
    print(f'  {r.image:<24}{r.ssim_seed_floor:>12.4f}{r.ssim_rand_shield:>14.4f}'
          f'{r.ssim_adv:>13.4f}   {"shield is real" if beats else "INDISTINGUISHABLE"}')

# Measured CLIP floor: how far apart two INNOCENT edits of the same image sit under CLIPScore.
# This is the honest P_FLOOR for D1 -- go back and set P_FLOOR to it, rather than guessing.
if 'clip_seed_gap' in d2 and d2.clip_seed_gap.notna().any():
    floor = float(d2.clip_seed_gap.abs().max())
    print(f'\n  measured CLIP seed floor = {floor:.2f} pts -- two INNOCENT edits of the same'
          f'\n  image differ by this much under CLIPScore. Set P_FLOOR = {floor:.2f} in D1 and'
          '\n  re-run it; any protection efficacy smaller than this is not protection.\n')

real = int((d2.ssim_adv < d2[["ssim_seed_floor","ssim_rand_shield"]].min(axis=1) - 0.03).sum())
print(f'\n  {real} of {len(d2)} images: the shield disturbs the edit more than a seed change'
      f' or a random perturbation of the same size does.\n')

if real == 0:
    print('VERDICT -- ssim_adv is measuring seed variance, not protection.')
    print('  Consequence: R_pipe has no valid denominator at these settings, so M2 must NOT')
    print('  claim a stronger wash. Two honest moves, in order:')
    print('    1. Publish this as the finding. "PhotoGuard at (40,2), pg_eps=16 does not engage')
    print('       this editor, and here are the two controls that show it." That is a')
    print('       measurement, and Limitation 1 already predicted pg_eps was the untested lever.')
    print('    2. Raise pg_eps (32, 64) and re-run D2 until the shield clears both nulls.')
    print('       THEN the R_pipe comparison between arms becomes meaningful.')
    print('  The fidelity claim -- C costs a quarter of A -- survives all of this untouched:')
    print('  it compares purified images to the clean photo and never uses the editor at all.')
else:
    print('VERDICT -- the shield clears both nulls. R_pipe has a real denominator.')
    print('  You may compare arms on R_pipe, subject to the +/-2-4 pp noise floor,')
    print('  and you now have the control that shows the denominator is not seed noise.')


---
## D4 · The seed floor for all TEN challenge images

**Run this section on its own.** It does not need D0, D1, D2, or any archived run — it never
looks at a shield. That is the whole point: **the floor is a property of the pipeline** (editor,
prompt, seed pair), which is identical for every team that enters. Measure it once, reuse it
forever.

### Why we need it

The published spec promises to report `R_pipe` only when
SSIM(edited-protected, edited-clean) ≤ **0.85**. That number was never measured. D2 then measured
the floor at **0.472 / 0.415** — and our own PhotoGuard shield sits at **0.584 / 0.610**.

**0.85 would pass a shield we have proven is indistinguishable from generator noise.** The gate
has to be the measured floor, and it has to be **per image**: our two differ by 0.06.

> The scale cannot resolve anything under five grams. You measure that once. But you still check
> where each parcel lands — a calibrated scale does not make a feather weighable.

### How to run it on Kaggle

1. New notebook → upload this `.ipynb`, or paste the cell below into a blank one.
2. **Accelerator = GPU T4 ×2** · **Internet = ON** (it fetches the Helen set and SD weights).
3. Run **only the cell below**. Skip D0–D3 entirely.
4. **Download `seed_floor_10.json` from the Output panel before closing the tab.**

**Cost: ~8 minutes.** Twenty edits at 50 steps, plus the model load. No protection and no
purification runs at all — those are where the hours go, and none of them are needed here.

It rebuilds the ten images from the same Drive archive with the same
`sorted(os.listdir(clean))[:10]` rule the pack used, so you do not need to upload anything.

In [ ]:
# D4 · seed floor for all ten pack images. SELF-CONTAINED — run this cell alone.
#      No archived run, no protect, no purify: the floor never depends on a shield.
import os, json, zipfile, pathlib, subprocess, importlib.util

PROMPT          = 'a person in an airplane'   # must match PARAMS in Dark_Tyro_M2.ipynb
TEST_DIFF_STEPS = 50
TEST_GUIDANCE   = 7.5
SEED_A, SEED_B  = 0, 1                        # two innocent edits of the SAME clean image
N_PACK          = 10

BASE = (pathlib.Path('/kaggle/working') if os.path.isdir('/kaggle/working')
        else pathlib.Path('/content') if os.path.isdir('/content') else pathlib.Path.cwd())

for mod, pkg in [('skimage', 'scikit-image'), ('gdown', 'gdown'), ('diffusers', 'diffusers')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run(f'pip install -q {pkg}', shell=True, check=False)

import numpy as np, torch, gdown
from PIL import Image, ImageOps
from skimage.metrics import structural_similarity as _ss

# Rebuild the pack from its source, by the pack's own selection rule. Nothing to upload.
ROOT, ZP = BASE / 'helen_face', BASE / 'helen_face_dataset.zip'
if not (ROOT / 'clean').exists():
    if not ZP.exists():
        gdown.download(id='16xISe7M_DlSqM2Zf2lWI4JJXcdsDEPsl', output=str(ZP), quiet=True)
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZP) as z: z.extractall(ROOT)
NAMES = sorted(os.listdir(ROOT / 'clean'))[:N_PACK]
assert len(NAMES) == N_PACK, (f'only {len(NAMES)} images in clean/. A previous run of '
                              'Dark_Tyro_M2 prunes it to N_IMAGES — delete helen_face/ and re-run.')

DEV = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if 'pipe' not in globals():                    # reuse D2a's pipe if this notebook already loaded one
    from diffusers import StableDiffusionInpaintPipeline
    for mid in ['runwayml/stable-diffusion-inpainting',
                'stable-diffusion-v1-5/stable-diffusion-inpainting']:
        try:
            pipe = StableDiffusionInpaintPipeline.from_pretrained(
                mid, torch_dtype=torch.float16 if DEV.startswith('cuda') else torch.float32,
                safety_checker=None).to(DEV)
            print('loaded', mid); break
        except Exception as e:
            print(f'  {mid} unavailable ({type(e).__name__})')
    assert 'pipe' in globals(), 'no inpainting checkpoint available'

def _edit(img, mask_rgb, seed):
    """Byte-for-byte the call pg_generate.test_image() makes."""
    np.random.seed(seed); torch.manual_seed(seed)
    if DEV.startswith('cuda'): torch.cuda.manual_seed(seed)
    return pipe(prompt=PROMPT, image=img, mask_image=mask_rgb, eta=1,
                num_inference_steps=TEST_DIFF_STEPS, guidance_scale=TEST_GUIDANCE).images[0]

_ssim = lambda a, b: float(_ss(np.asarray(a, np.float64), np.asarray(b, np.float64),
                               channel_axis=2, data_range=255))

floors = {}
for i, n in enumerate(NAMES, 1):
    clean = Image.open(ROOT / 'clean' / n).convert('RGB').resize((512, 512))
    # pg_generate INVERTS the mask before the pipeline sees it. Copy that, do not re-derive it.
    mask  = ImageOps.invert(Image.open(ROOT / 'mask' / n).convert('RGB').resize((512, 512)))
    floors[n] = round(_ssim(_edit(clean, mask, SEED_A), _edit(clean, mask, SEED_B)), 4)
    print(f'  [{i:>2}/{N_PACK}] {n:<24} seed floor = {floors[n]:.4f}', flush=True)

out = BASE / 'seed_floor_10.json'
out.write_text(json.dumps(dict(prompt=PROMPT, seeds=[SEED_A, SEED_B],
                               test_diff_steps=TEST_DIFF_STEPS, test_guidance=TEST_GUIDANCE,
                               floors=floors), indent=2))
v = np.array(list(floors.values()))
print(f'\n  n={len(v)}   min {v.min():.4f}   median {np.median(v):.4f}   max {v.max():.4f}')
print(f'  -> {out}   ** DOWNLOAD THIS from the Output panel before closing the tab. **')
print('\n  THE GATE: report R_pipe for an entry only when its ssim_adv < the floor for THAT')
print('  image. Our own PhotoGuard shield scores 0.584 / 0.610, which is ABOVE the floor, so')
print('  it does not engage — and the old published threshold of 0.85 would have passed it.')


---
## D3 · Testing a different `pg_eps` — ⛔ SUPERSEDED 9 Sep

> **Do not run this.** We executed it at `pg_eps = 32` and the perturbation came out the same
> as at 16, to within 1% (L2 2645.8 → 2641.6). At `pg_iters=40, pg_step_size=1` the L2 ball is
> never reached, so `pg_eps` never binds — the knob is inert on this code path. The lever that
> binds is `pg_step_size`, and that is M3 work. Kept below as the record of what we tried.

**D2 has no `pg_eps` setting on purpose.** It never protects an image; it reads `clean512/` and
`protected/` out of the archive you point `RUN_DIR` at. So to test a stronger shield you have to
*make* one first, in `Dark_Tyro_M2.ipynb`, and then point D2 at the new archive.

You do **not** need a full four-arm run. The `N_no_wash` arm alone produces everything D2 needs
(`clean512/`, `protected/`, `edit_clean/`, `edit_protected/`) because it skips purification
entirely.

### In `Dark_Tyro_M2.ipynb`, change two things

```python
# cell 4 (PARAMS)
PARAMS = dict(..., pg_eps=32, ...)          # <- 16 -> 32, then 64

# cell 4 (ARMS) -- ONE arm, and name it after the eps so runs do not overwrite each other
ARMS = [dict(name=f"N_eps{PARAMS['pg_eps']}", pur_iters=None, masked=False)]

SMOKE = False
```

Then run cells 1-7 and stop. Skip cells 8 and 9 (the scoreboard and the charts) — they expect
several arms and there is only one.

### Cost

| stage | 2 images |
|---|---|
| protect at `(40, 2)` | ~12 min |
| edit (3 variants) | ~1 min |
| **then D2** | ~3 min |

**≈ 15 min per `pg_eps` value.** No purification runs at all, which is where the time normally
goes.

### Back in this notebook

Set `RUN_DIR` to the new archive folder and re-run D0 and D2. Keep **one eps per folder** — D2
takes `clean512/` and `protected/` from `ARMS[0]`, so mixing two eps values in one folder will
silently compare the wrong pair.

### Reading it

You are looking for **`ssim_adv` to drop clearly below `ssim_seed_floor`** — the shield finally
disturbing the edit more than a seed change does. That is the moment `R_pipe` acquires a real
denominator and the arm comparison becomes worth running.

> Raising `pg_eps` makes the *defence stronger*, not weaker. You can only measure an attack
> against a lock that actually locks.


---
## What to do with these two results

| D2 says | what M2 claims |
|---|---|
| shield **clears** both nulls | *"C matches A's attack strength to within noise, at a quarter of the perceptual cost."* A **cost** claim, not a strength claim — the arm gaps are still smaller than the between-image spread at n = 2. |
| shield is **indistinguishable** | *"PhotoGuard at `(40, 2)`, `pg_eps = 16` does not engage this editor — here are two controls that prove it — so no purifier can be scored against it. Our mask-restricted wash still cuts IMPRESS's fidelity cost by 61-63%, measured without the editor."* |

Either row is a defensible M2. Neither requires the result to have gone your way, which is the
whole reason to run the controls before the milestone rather than after it.

**The fidelity half never depends on this.** SSIM/LPIPS(purified, clean) compares two still
images and never invokes the diffusion editor, which is exactly why it reproduced to four
decimals across runs while `R_pipe` wandered.
